In [0]:
# Databricks notebook source
# Step 3: Bronze layer — load raw CSVs into Delta tables
# Run in a new notebook: 02_bronze_layer
 
CATALOG = "workspace"          # change if your catalog has a different name
SCHEMA = "fifa_project"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"
 
# Make sure the schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
 
# Tables from the European Soccer Database (skip sqlite_sequence — it's a system table)
tables = ["Country", "League", "Match", "Player", "Player_Attributes", "Team", "Team_Attributes"]
 
for table in tables:
    file_path = f"{VOLUME_PATH}/{table}.csv"
    print(f"Loading {table}...")
 
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(file_path)
    )
 
    bronze_table = f"{CATALOG}.{SCHEMA}.bronze_{table.lower()}"
    df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)
 
    print(f"  -> {bronze_table}: {df.count():,} rows, {len(df.columns)} columns")
 
print("\nBronze layer complete!")
 
# COMMAND ----------
 
# Quick check — list all bronze tables
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA} LIKE 'bronze_*'"))

In [0]:
df = spark.table("fifa_project.bronze_match")
display(df.count())

In [0]:
df = spark.table("fifa_project.bronze_player")
display(df.count())